## Behavorial Analysis With BIDSReader
The **BIDSReader** package helps CML researchers efficiently load BIDS data without having to deal with the BIDS syntax. This introduction will teach you how to load behavioral BIDS data using **BIDSReader** for analysis.

### Loading Study-wide Metadata
Once you set the study root, you can inquire simple metadata queries such a getting a list of subjects, tasks, and max number of sessions in the study.

In [1]:
# imports
import pandas as pd; pd.set_option('display.max_columns', None)
import numpy as np
import os 
import sys
sys.path.insert(0, 'dependencies/bidsreader')
from bidsreader import CMLBIDSReader as BIDSReader

sys.path.insert(0, ".")
from cml_data import (get_bids_root, available_subjects, available_tasks,
                      session_dataframe)

# The PEERS scalp-EEG dataset lives on OpenNeuro as ds004395. Ask OpenNeuro
# what it contains without downloading anything:
print(f"Scalp subjects: {len(available_subjects('ltpFR2'))} "
      f"(first few: {available_subjects('ltpFR2')[:5]})")
print(f"Tasks for LTP093: {available_tasks('ltpFR2', 'LTP093')}")

# Point the reader at one session (cluster copy, or downloaded on request).
bids_root = get_bids_root("ltpFR2", subject="LTP093", session=0)
reader = BIDSReader(root=bids_root, subject="LTP093", session=0, task="ltpFR2")
print(f"Sessions cached for this subject: {reader.get_subject_sessions()}")

Scalp subjects: 364 (first few: ['LTP063', 'LTP064', 'LTP065', 'LTP066', 'LTP067'])
Tasks for LTP093: ['ltpFR', 'ltpFR2']


ds004395 (ltpFR2): already downloaded -> bids_data/ds004395
Sessions cached for this subject: ['0']


### The two-step loading pattern

Every time you load data in this course, it is the same two steps:

```python
root   = get_bids_root(task, subject=sub, session=ses)   # 1. make sure the files are here
reader = BIDSReader(root=root, subject=sub, task=task, session=ses)   # 2. read them
evs    = reader.load_events()
```

**Step 1 is not optional.** `get_bids_root` returns the cluster copy if you are on
Rhino, and otherwise downloads just that session from OpenNeuro (a few hundred KB
for behavioural data) and caches it. If you skip it and point `BIDSReader` at a
folder that has nothing in it yet, you get a confusing "file does not exist" error.

**To ask what data exists**, use `session_dataframe(task)` rather than scanning the
folder. Scanning only sees what you have already downloaded:

```python
df = session_dataframe("ltpFR2")   # every subject/session, from OpenNeuro
len(df), df["subject"].nunique()
```

### Loading Subject Metadata
You can use those inquiries to choose a subject and inquire about the tasks and sessions the subject has performed.

In [2]:
subject = "LTP093"

# use set_fields to set multiple fields at once
reader.set_fields(subject=subject)
tasks_list = reader.get_subject_tasks()
sessions_list = reader.get_subject_sessions()
print(f"{subject} Tasks: {tasks_list}")
print(f"{subject} Sessions: {sessions_list}")

LTP093 Tasks: ['ltpFR2']
LTP093 Sessions: ['0']


## Example of some of the tasks ran by the lab by category:

> Only some of these are published on OpenNeuro so far: **FR1**, **catFR1**,
> **PAL1**, **pyFR**, **RepFR1** (intracranial) and **ltpFR / ltpFR2 / VFFR**
> (PEERS scalp), plus **NICLS**. The rest are currently cluster-only.

### Verbal free-recall tasks (no-stim)
* FR1
* catFR1
* RepFR1

### Paired-associates tasks
* PAL1
* PAL2 (open-loop stim)
* PAL3 (closed-loop stim)
* PAL5 (closed-loop stim)

### Spatial navigation tasks
* YC1
* TH1
* THR
* THR1
* YC2 (open-loop stim)
* TH3 (closed-loop stim)
* EFRCourierReadOnly
* EFRCourierOpenLoop

### Verbal free-recall w/ stim
(Basically, any FR task with a number above 1 somewhere)
* FR2 (open-loop)
* catFR2
* FR3 (closed-loop)
* catFR3
* FR5 (closed-loop)
* catFR5
* PS4_FR (closed-loop)
* PS4_catFR (closed-loop)
* PS5_catFR (closed-loop)
* FR6 (multi-target stim)
* catFR6 (multi-target stim)
* TICL_FR (encoding/math/retrieval stim)
* RepFR2

### No-task stimulation ("parameter search")
* PS1
* PS2/PS2.1
* PS3
* LocationSearch
* OPS


## Loading Behavioral Events
After identifying the tasks and sessions completed by your chosen subject, you can now load the events.

### BIDSReader Fields
* root (str | Path: path to the study root) (required; see `get_bids_root` above)
* subject (str: subject id)
* task (str: experiment id)
* session (str: session id)
* eeg_type (str: type of eeg used) (eeg or ieeg)
* acquisition (str: type of ieeg acquisition) (bipolar or monopolar)
* space (str: iEEG coordinate system) (MNI152NLin6ASym, Talarich) 

To load events, you must specify the subject, task, and session. eeg_type and space are necessary for BIDS, but will be infered based on the existing data.

In [3]:
# set task and session which will let BIDSReader infer eeg_type and space 
task = tasks_list[0]
session = sessions_list[0]
reader.set_fields(task=task, session=session)

CMLBIDSReader(root=PosixPath('bids_data/ds004395'), subject='LTP093', session='0', task='ltpFR2', device='eeg', space='CapTrak')

## Loading Behavioral Data
As mentioned, there are two locations where the behavioral data is held, in the "beh" folder and in the "eeg/ieeg" folder. BIDSReader makes it easy to load from each by setting "event_type" in the "load_events" function to "beh" or "events". 

In [4]:
evs_beh = reader.load_events(event_type="beh")
evs_beh[:5]

,mstime,trial_type,stim_file,subject,experiment,session,trial,item_name,item_num,list,answer,test_x,test_y,test_z
0,0,SESS_START,NaN,LTP093,ltpFR2,0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0
1,82245,START,NaN,LTP093,ltpFR2,0,NaN,NaN,NaN,-1.0,NaN,NaN,NaN,NaN
2,82277,PROB,NaN,LTP093,ltpFR2,0,NaN,NaN,NaN,-1.0,24.0,7.0,8.0,9.0
3,87951,PROB,NaN,LTP093,ltpFR2,0,NaN,NaN,NaN,-1.0,20.0,3.0,8.0,9.0
4,90864,PROB,NaN,LTP093,ltpFR2,0,NaN,NaN,NaN,-1.0,11.0,2.0,6.0,3.0


In [5]:
evs_eeg = reader.load_events(event_type="eeg")
evs_eeg[:5]

,onset,duration,trial_type,sample,stim_file,subject,experiment,session,trial,item_name,item_num,list,answer,test_x,test_y,test_z
0,416.672,NaN,SESS_START,208336,NaN,LTP093,ltpFR2,0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0
1,498.918,NaN,START,249459,NaN,LTP093,ltpFR2,0,NaN,NaN,NaN,-1.0,NaN,NaN,NaN,NaN
2,498.950,NaN,PROB,249475,NaN,LTP093,ltpFR2,0,NaN,NaN,NaN,-1.0,24.0,7.0,8.0,9.0
3,504.624,NaN,PROB,252312,NaN,LTP093,ltpFR2,0,NaN,NaN,NaN,-1.0,20.0,3.0,8.0,9.0
4,507.536,NaN,PROB,253768,NaN,LTP093,ltpFR2,0,NaN,NaN,NaN,-1.0,11.0,2.0,6.0,3.0


## evs/Events Data Frame

Before digging into the data, a little information about the experiment.

In free recall experiments, subjects memorize a series of word lists during an experimental session. Each list consists of an encoding period, during which the words (or "items") in the list are presented to the subject one by one, followed by a retrieval period during which the subject recalls as many words as they can in any order. The experiment is called "free" recall because the subjects are "free" to recall the items in any order; this in contrast to a serial recall experiment in which subjects must recall the items in the order they were presented.

Some free recall experiments contain a "distractor" period between the end of the encoding period and the beginning of the retrieval period. The purpose of the distractor is to "clear out" subjects' minds before starting to recall items; without the distractor, subjects are far more likely to recall items from the end of the list in what is known as the recency effect. In the experiments described here, the distractor period consists of a series of arithmetic problems of the form 'A + B + C', to which subjects respond by typing the answer. The FR1 experiment also begins each list by presenting the subject with a countdown period (10, 9, 8, ...).

The events dataframe contains information about everything that happened during an experimental session. It indicates the time at which every word appeared on the screen, and when those words were later recalled. It also contains information about events that you might not care about, such as when the countdown timer starts and ends.
<center>
<img src="https://github.com/esolomon/PythonBootcamp2019/blob/master/figures/task_design-01.jpg?raw=true" width=650>
</center>
Let's take a look at all the columns in this dataframe...

In [6]:
cols = set(evs_beh.columns) | set(evs_eeg.columns)
print(sorted(cols))

['answer', 'duration', 'experiment', 'item_name', 'item_num', 'list', 'mstime', 'onset', 'sample', 'session', 'stim_file', 'subject', 'test_x', 'test_y', 'test_z', 'trial', 'trial_type']


... and here is what the important ones mean.

**Timing** — the same event is described two ways depending on which file you loaded (see above):
* `mstime` — time in milliseconds from the experiment laptop. Only in the **beh** version. Good for comparing events to each other; the absolute value is meaningless.
* `onset` — time in seconds from the start of the EEG recording. Only in the **eeg/ieeg** version.
* `sample` — the same instant expressed in samples of the EEG file. This is what lets you cut EEG around an event; MNE-BIDS needs it, but you rarely touch it directly.
* `duration` — length of the event in seconds. Currently `NaN` throughout.

**What happened**
* `trial_type` — the kind of event, e.g. `WORD` (a word shown during encoding) or `REC_WORD` (a word spoken during recall).
* `item_name` — the word that was presented or recalled.
* `item_num` — the word's ID in the word pool. This is what you match on to decide whether a recalled word was one that was actually presented.
* `answer`, `test_x`/`test_y`/`test_z` — the subject's response to an arithmetic distractor problem, and its three operands.
* `experiment`, `session`, `subject` — which task, session, and subject the row came from.

**Which list a row belongs to**
* **`list` is the list number — except in the PEERS scalp datasets (`ltpFR`, `ltpFR2`), where the list number is in the `trial` column instead.** `ltpFR2` happens to have both; `ltpFR` has only `trial`, and `FR1` has only `list`. Always check `evs.columns` before assuming.

**Columns you have to build yourself**

Several quantities you will need every week are *not* stored in the BIDS events, because they are derived rather than recorded. Expect to compute these:

| Quantity | Present in the data? | How you get it |
|---|---|---|
| `serialpos` — position of a word within its list | Only in `FR1` | `groupby(list).cumcount() + 1` over the `WORD` rows |
| `recalled` — was this word later recalled? | **Never** | Match each `WORD` against the `REC_WORD` rows *of the same list* |
| `intrusion` — was a recalled word not from this list? | **Never** | A `REC_WORD` whose `item_num` is not in the list just studied |
| `distractor` — length of the distractor period | **Never** | Recover from event timing (you will do this in Assignment 1) |

This is the single most common source of confusion in the next assignment: reach for a column, get a `KeyError`, and assume you have made a mistake. You have not — the column simply does not exist, and building it is the exercise.

For the authoritative description of every column in a given dataset, read the `*_events.json` sidecar that sits next to the events file. Each BIDS dataset ships its own, and it is the definitive source for that dataset.

The two event types we analyse most are encoding events (`WORD`) and recall events (`REC_WORD`).

`WORD` is every word shown during encoding. `REC_WORD` is every word the subject **said** during the recall period — which is not the same thing as every word they got *right*. Some recalls are:

* **prior-list intrusions** — a word that was presented, but on an earlier list;
* **extra-list intrusions** — a word that was never presented at all;
* **vocalisations** (`REC_WORD_VV`) — a sound that was not a word. These get their own `trial_type`, so they are easy to exclude.

This matters more than it looks. A recall rate computed as `len(rec_evs) / len(word_evs)` counts intrusions as successes and comes out too high. To do it correctly, match each recalled word back to a presented word **on the same list** — pairing on `(list, item_num)`, not on `item_num` alone.

Intrusions are rare in a good session, so the shortcut gives an answer that looks plausible. That is exactly what makes it worth being careful about.

In [7]:
# let's filter for encoding events
word_evs = evs_beh[evs_beh['trial_type'] == 'WORD']
word_evs.head()

,mstime,trial_type,stim_file,subject,experiment,session,trial,item_name,item_num,list,answer,test_x,test_y,test_z
6,155185,WORD,wordpools/wasnorm_wordpool_576.txt,LTP093,ltpFR2,0,1.0,BALLOON,75.0,NaN,NaN,0.0,0.0,0.0
7,157680,WORD,wordpools/wasnorm_wordpool_576.txt,LTP093,ltpFR2,0,1.0,MAILBOX,857.0,NaN,NaN,0.0,0.0,0.0
8,160211,WORD,wordpools/wasnorm_wordpool_576.txt,LTP093,ltpFR2,0,1.0,FLOWER,584.0,NaN,NaN,0.0,0.0,0.0
9,162956,WORD,wordpools/wasnorm_wordpool_576.txt,LTP093,ltpFR2,0,1.0,DAUGHTER,442.0,NaN,NaN,0.0,0.0,0.0
10,165685,WORD,wordpools/wasnorm_wordpool_576.txt,LTP093,ltpFR2,0,1.0,BLUEPRINT,142.0,NaN,NaN,0.0,0.0,0.0


In [8]:
# as well as recall events
rec_evs = evs_beh[evs_beh['trial_type'] == 'REC_WORD']
rec_evs.head()

,mstime,trial_type,stim_file,subject,experiment,session,trial,item_name,item_num,list,answer,test_x,test_y,test_z
44,244119,REC_WORD,wordpools/wasnorm_wordpool_576.txt,LTP093,ltpFR2,0,1.0,BALLOON,75.0,NaN,NaN,0.0,0.0,0.0
45,244816,REC_WORD,wordpools/wasnorm_wordpool_576.txt,LTP093,ltpFR2,0,1.0,MAILBOX,857.0,NaN,NaN,0.0,0.0,0.0
46,245784,REC_WORD,wordpools/wasnorm_wordpool_576.txt,LTP093,ltpFR2,0,1.0,FLOWER,584.0,NaN,NaN,0.0,0.0,0.0
47,246916,REC_WORD,wordpools/wasnorm_wordpool_576.txt,LTP093,ltpFR2,0,1.0,DAUGHTER,442.0,NaN,NaN,0.0,0.0,0.0
48,248664,REC_WORD,wordpools/wasnorm_wordpool_576.txt,LTP093,ltpFR2,0,1.0,BLUEPRINT,142.0,NaN,NaN,0.0,0.0,0.0


**Exercise (optional): What is R1171M's overall recall percentage on FR1?**

Note that R1171M is an **intracranial** subject, so you need a second reader pointed at the FR1 dataset rather than the PEERS one you have been using:

```python
fr1_root = get_bids_root("FR1", subject="R1171M", session=0)
fr1 = BIDSReader(root=fr1_root, subject="R1171M", session=0, task="FR1")
evs = fr1.load_events(event_type="beh")
```

Two things to watch, both covered above: FR1 calls the list number `list` (not `trial`), and `REC_WORD` includes intrusions, so match recalls back to presented words on the same list.

**Exercise (optional): What is R1171M's recall percentage at each serial position on FR1? Plot your result.**

FR1 already carries a `serialpos` column, so you can group by it directly — you will not be so lucky in the next assignment, which uses the PEERS scalp data and makes you derive it.

The shape you are looking for is the **serial position curve**: high at the start of the list (primacy), high at the end (recency), and lowest in the middle.